# DeepSeek-V4-Mini Lecture 2: Multi-Query-Attention 

1. MHA
2. MQA
3. MHA absord to MQA

## config & data

In [6]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
torch.manual_seed(42)

In [7]:
# config 
@dataclass
class AttentionModelArgs:
    dim: int = 512
    head_dim: int = 64
    n_heads: int = 8
    n_kv_heads: int = 8
    
args = AttentionModelArgs()

# data
bsz = 2
seq_len = 256
X = torch.randn(bsz, seq_len, args.dim)

## MHA

In [10]:
def ScaledDotProductionAttention(Q, K, V, M):
    """
    argps
        input: [B, H, L, D] or [B, L, D]
        output: [B, H, L, D] or [B, L, D]
    """
    S = Q @ K.transpose(-2, -1)
    if M != None:
        S = S.masked_fill(M == False, float('-inf'))
    P = F.softmax(S, dim = -1)
    Z = P @ V
    return Z

class Attention(nn.Module):
    """
        single head attention
    """
    def __init__(self, args):
        super().__init__()
        dim, head_dim, self.n_heads, self.n_kv_heads = args.dim, args.head_dim, args.n_heads, args.n_kv_heads
        self.Wq = nn.Linear(dim, self.n_heads * head_dim, bias=False)
        self.Wk = nn.Linear(dim, self.n_kv_heads * head_dim, bias=False)
        self.Wv = nn.Linear(dim, self.n_kv_heads * head_dim, bias=False)
        self.Wo = nn.Linear(dim, dim, bias=False)

    def forward(self, X, mask):
        B,L,D = X.shape
        Q, K, V = self.Wq(X), self.Wk(X), self.Wv(X)
        Q = Q.reshape(B, L, self.n_heads, D//self.n_heads).transpose(1,2)
        K = K.reshape(B, L, self.n_heads, D//self.n_kv_heads).transpose(1,2)
        V = V.reshape(B, L, self.n_heads, D//self.n_kv_heads).transpose(1,2)
        Z = ScaledDotProductionAttention(Q, K, V, mask)
        Z = Z.reshape(B, L, D)
        O = self.Wo(Z)
        return O
        
MHA = Attention(args)
O = MHA(X, None)
print(O.shape)

torch.Size([2, 256, 512])


## MQA

In [11]:
args.n_kv_head = 1
MQA = Attention(args)
O = MQA(X, None)
print(O.shape)

torch.Size([2, 256, 512])


### MHA absord to MQA

In [ ]:
class MQA(nn.Module):
    """
       Deepseek-V4 MQA
    """
    def __init__(self, args, o_lora_rank, o_groups):
        super().__init__()
        dim, head_dim, self.n_heads = args.dim, args.head_dim, args.n_heads
        self.n_kv_heads = 1
        self.Wq = nn.Linear(dim, self.n_heads * head_dim, bias=False)
        self.Wc = nn.Linear(dim, head_dim, bias=False)
        self.Wk = nn.Linear(head_dim, head_dim, bias=False)
        self.Wv = nn.Linear(head_dim, head_dim, bias=False)
        
        # self.Wo = nn.Linear(self.n_heads * head_dim, dim, bias=False)
        self.Wo_lora = nn.Linear(self.n_heads * head_dim // o_lora_rank, o_lora_rank * o_groups, bias=False)
        self.Wo = nn.Linear(o_lora_rank * self.n_heads, dim, bias=False)
        
        self.o_lora_rank, self.o_groups = o_lora_rank, o_groups
        

    def forward(self, X, mask):
        B,L,D = X.shape
        Q, K, V = self.Wq(X), self.Wk(X), self.Wv(X)
        Q = Q.reshape(B, L, self.n_heads, D//self.n_heads).transpose(1,2)
        K = K.reshape(B, L, self.n_heads, D//self.n_kv_heads).transpose(1,2)
        V = V.reshape(B, L, self.n_heads, D//self.n_kv_heads).transpose(1,2)
        
        Z = ScaledDotProductionAttention(Q, K, V, mask) # -> B, H, L, D
        
        # [B, H, L, D] -> [B, L, H, D] -> [B, L, G, d]
        d = self.n_heads * D / self.o_groups
        self.Wo_lora.weight.view(self.o_groups, self.o_lora_rank, d) # 三维参数层怎么处理
        Z.reshape(B, L, self.o_groups, d)
        Z = self.Wo_lora(Z)
        Z.reshape(B, L, self.o_lora_rank*d)
        O = self.Wo(Z)
        
        return O
        
MHA = Attention(args)
O = MHA(X, None)
print(O.shape)